# peekabook-crs-test7 결과 분석

wandb 프로젝트 `peekabook-crs-test7`에서 실험 결과를 불러와  
NDCG@3,5,10 / Hit-rate@3,5,10 / 평균 점수 / 평균 비용을 계산

**비교축**: `query_transform` (none vs hyde) × `collection_name` × `use_genre_filter`

**전제**: 각 런에 `book_1_score` ~ `book_10_score` (0 또는 1) 가 기록되어 있어야 함

In [9]:
import math
import pandas as pd
import wandb

# ── 설정 ─────────────────────────────────────────────────────────────────────
WANDB_PROJECT = "jjeong3150-aiffel/peekabook-crs-test7-2"
N_BOOKS       = 10   # 리랭킹 후 평가 대상 권수

# 비용 계산용 단가 (USD per 1M tokens)
MODEL_COSTS = {
    "gpt-4o-mini": (0.15,  0.60),
    "gpt-4o":      (2.50, 10.00),
}
ACTIVE_MODEL = "gpt-4o-mini"

In [10]:
# ── 지표 계산 함수 ─────────────────────────────────────────────────────────

def dcg_at_k(scores: list, k: int) -> float:
    return sum(r / math.log2(i + 2) for i, r in enumerate(scores[:k]))

def ndcg_at_k(scores: list, k: int) -> float:
    dcg  = dcg_at_k(scores, k)
    idcg = dcg_at_k(sorted(scores, reverse=True), k)
    return round(dcg / idcg, 4) if idcg > 0 else 0.0

def hit_rate_at_k(scores: list, k: int) -> int:
    return 1 if any(s > 0 for s in scores[:k]) else 0

def compute_metrics(ranked_scores: list) -> dict:
    n = len(ranked_scores)
    return {
        "ndcg@3":        ndcg_at_k(ranked_scores, 3),
        "ndcg@5":        ndcg_at_k(ranked_scores, 5),
        "ndcg@10":       ndcg_at_k(ranked_scores, 10),
        "hr@3":          hit_rate_at_k(ranked_scores, 3),
        "hr@5":          hit_rate_at_k(ranked_scores, 5),
        "hr@10":         hit_rate_at_k(ranked_scores, 10),
        "mean_score@3":  round(sum(ranked_scores[:3])  / 3,  4) if n >= 3  else 0.0,
        "mean_score@5":  round(sum(ranked_scores[:5])  / 5,  4) if n >= 5  else 0.0,
        "mean_score@10": round(sum(ranked_scores[:10]) / 10, 4) if n >= 10 else 0.0,
    }

def tokens_to_cost(input_tokens: int, output_tokens: int, model: str = ACTIVE_MODEL) -> float:
    in_rate, out_rate = MODEL_COSTS.get(model, MODEL_COSTS["gpt-4o-mini"])
    return (input_tokens * in_rate + output_tokens * out_rate) / 1_000_000

In [11]:
# ── wandb 런 로드 ──────────────────────────────────────────────────────────

api  = wandb.Api()
runs = api.runs(WANDB_PROJECT)

records = []
for run in runs:
    s   = run.summary
    cfg = run.config

    scores = []
    for i in range(1, N_BOOKS + 1):
        v = s.get(f"book_{i}_score")
        if v is None:
            break
        scores.append(int(v))

    if not scores:
        continue

    cost = tokens_to_cost(
        int(s.get("total_input_tokens",  0)),
        int(s.get("total_output_tokens", 0)),
    )

    records.append({
        "run_id":            run.id,
        "run_name":          run.name,
        "persona_name":      cfg.get("persona_name",     s.get("persona_name", "")),
        "query_transform":   cfg.get("query_transform",  ""),
        "use_genre_filter":  cfg.get("use_genre_filter", ""),
        "collection_name":   cfg.get("collection_name",  ""),
        "session_id":        s.get("session_id", ""),
        "n_books_evaluated": len(scores),
        **compute_metrics(scores),
        "total_input_tokens":  int(s.get("total_input_tokens",  0)),
        "total_output_tokens": int(s.get("total_output_tokens", 0)),
        "estimated_cost_usd":  round(cost, 6),
    })

df = pd.DataFrame(records)
print(f"로드된 런: {len(df)}개")
df.head()

로드된 런: 135개


,run_id,run_name,persona_name,query_transform,use_genre_filter,collection_name,session_id,n_books_evaluated,ndcg@3,ndcg@5,ndcg@10,hr@3,hr@5,hr@10,mean_score@3,mean_score@5,mean_score@10,total_input_tokens,total_output_tokens,estimated_cost_usd
0,a87yu0i3,super-sweep-1,A_최재원,none,False,books_intro_48k,f1906171,10,0.7654,0.8304,0.9420,1,1,1,0.6667,0.8000,0.7000,12682,2078,0.0031
1,rg72fnm7,solar-sweep-2,A_최재원,none,False,books_intro_48k,2c1d69ac,10,0.5307,0.4415,0.7107,1,1,1,0.6667,0.4000,0.4000,11012,1822,0.0027
2,lxug357a,playful-sweep-3,A_최재원,none,False,books_intro_48k,749274c1,10,0.7654,0.6367,0.8843,1,1,1,0.6667,0.4000,0.4000,12667,2124,0.0032
3,7cadp0wt,bright-sweep-4,A_최재원,none,False,books_intro_48k,e9f2b14d,9,0.7039,0.7860,0.9002,1,1,1,0.6667,0.8000,0.0000,11614,1933,0.0029
4,0wv9pc5y,misty-sweep-5,A_최재원,none,False,books_intro_48k,83c233a8,10,1.0000,1.0000,1.0000,1,1,1,1.0000,1.0000,0.5000,13679,2047,0.0033


In [12]:
# ── 조건별 집계 ────────────────────────────────────────────────────────────

GROUP_COLS = ["collection_name", "use_genre_filter", "query_transform"]

summary = (
    df.groupby(GROUP_COLS)
    .agg(
        runs             = ("run_id",             "count"),
        ndcg_3           = ("ndcg@3",             "mean"),
        ndcg_5           = ("ndcg@5",             "mean"),
        ndcg_10          = ("ndcg@10",            "mean"),
        hr_3             = ("hr@3",               "mean"),
        hr_5             = ("hr@5",               "mean"),
        hr_10            = ("hr@10",              "mean"),
        mean_score_top3  = ("mean_score@3",       "mean"),
        mean_score_top5  = ("mean_score@5",       "mean"),
        mean_score_top10 = ("mean_score@10",      "mean"),
        avg_cost_usd     = ("estimated_cost_usd", "mean"),
    )
    .round(4)
    .reset_index()
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
summary

,collection_name,use_genre_filter,query_transform,runs,ndcg_3,ndcg_5,ndcg_10,hr_3,hr_5,hr_10,mean_score_top3,mean_score_top5,mean_score_top10,avg_cost_usd
0,books_intro_48k,False,decompose,45,0.4662,0.5408,0.7322,0.8222,0.9111,1.0000,0.4519,0.4533,0.3867,0.0026
1,books_intro_48k,False,hyde,45,0.6736,0.6983,0.8568,0.9333,1.0000,1.0000,0.5852,0.4978,0.4200,0.0027
2,books_intro_48k,False,none,45,0.6570,0.6566,0.8340,1.0000,1.0000,1.0000,0.6222,0.5200,0.4289,0.0028


In [13]:
# ── none vs hyde 유의성 검정 ───────────────────────────────────────────────
from scipy import stats

METRIC = "ndcg@10"

for collection in df["collection_name"].unique():
    for genre_filter in df["use_genre_filter"].unique():
        subset = df[
            (df["collection_name"]  == collection) &
            (df["use_genre_filter"] == genre_filter)
        ]
        a = subset[subset["query_transform"] == "none"][METRIC]
        b = subset[subset["query_transform"] == "hyde"][METRIC]

        if len(a) == 0 or len(b) == 0:
            continue

        t, p = stats.ttest_ind(a, b)
        print(f"[{collection} | genre_filter={genre_filter}]")
        print(f"  none: mean={a.mean():.4f}, n={len(a)}")
        print(f"  hyde: mean={b.mean():.4f}, n={len(b)}")
        print(f"  t={t:.4f}, p={p:.4f} {'✅ 유의' if p < 0.05 else '❌ 비유의'}\n")

[books_intro_48k | genre_filter=False]
  none: mean=0.8340, n=45
  hyde: mean=0.8568, n=45
  t=-0.9029, p=0.3690 ❌ 비유의



In [14]:
# ── 페르소나별 none vs hyde 비교 ──────────────────────────────────────────

persona_summary = (
    df.groupby(["persona_name", "query_transform"])
    .agg(
        runs    = ("run_id",    "count"),
        ndcg_3  = ("ndcg@3",   "mean"),
        ndcg_5  = ("ndcg@5",   "mean"),
        ndcg_10 = ("ndcg@10",  "mean"),
        hr_3    = ("hr@3",     "mean"),
        hr_5    = ("hr@5",     "mean"),
        hr_10   = ("hr@10",    "mean"),
    )
    .round(4)
    .reset_index()
)
persona_summary

,persona_name,query_transform,runs,ndcg_3,ndcg_5,ndcg_10,hr_3,hr_5,hr_10
0,A_최재원,decompose,15,0.4007,0.5013,0.6901,0.8000,0.8667,1.0000
1,A_최재원,hyde,15,0.5865,0.5639,0.8032,0.9333,1.0000,1.0000
2,A_최재원,none,15,0.6238,0.6153,0.8178,1.0000,1.0000,1.0000
3,B_한미영,decompose,15,0.3001,0.3901,0.6469,0.6667,0.8667,1.0000
4,B_한미영,hyde,15,0.5844,0.6220,0.8273,0.8667,1.0000,1.0000
5,B_한미영,none,15,0.6729,0.6510,0.8322,1.0000,1.0000,1.0000
6,C_오민아,decompose,15,0.6980,0.7310,0.8596,1.0000,1.0000,1.0000
7,C_오민아,hyde,15,0.8500,0.9090,0.9401,1.0000,1.0000,1.0000
8,C_오민아,none,15,0.6741,0.7036,0.8521,1.0000,1.0000,1.0000


In [15]:
# ── 런별 상세 (ndcg@10 내림차순) ─────────────────────────────────────────

detail_cols = [
    "persona_name", "query_transform", "use_genre_filter", "collection_name",
    "ndcg@3", "ndcg@5", "ndcg@10",
    "hr@3", "hr@5", "hr@10",
    "mean_score@10", "estimated_cost_usd", "session_id",
]
df[detail_cols].sort_values("ndcg@10", ascending=False).reset_index(drop=True)

,persona_name,query_transform,use_genre_filter,collection_name,ndcg@3,ndcg@5,ndcg@10,hr@3,hr@5,hr@10,mean_score@10,estimated_cost_usd,session_id
0,A_최재원,none,False,books_intro_48k,1.0000,1.0000,1.0000,1,1,1,0.5000,0.0033,83c233a8
1,C_오민아,hyde,False,books_intro_48k,1.0000,1.0000,1.0000,1,1,1,0.3000,0.0022,5a6642ae
2,C_오민아,hyde,False,books_intro_48k,1.0000,1.0000,1.0000,1,1,1,0.2000,0.0027,507ab938
3,C_오민아,none,False,books_intro_48k,1.0000,1.0000,1.0000,1,1,1,0.2000,0.0023,3a5c8d02
4,C_오민아,hyde,False,books_intro_48k,1.0000,1.0000,1.0000,1,1,1,0.2000,0.0023,cf5418b1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,A_최재원,decompose,False,books_intro_48k,0.0000,0.0000,0.4030,0,0,1,0.2000,0.0027,9d097522
131,B_한미영,decompose,False,books_intro_48k,0.0000,0.0000,0.3978,0,0,1,0.2000,0.0030,93086d47
132,B_한미영,decompose,False,books_intro_48k,0.0000,0.3869,0.3869,0,1,1,0.0000,0.0030,6665f66d
133,A_최재원,decompose,False,books_intro_48k,0.0000,0.0000,0.3780,0,0,1,0.2000,0.0026,25f4a36f


In [19]:
# ── HyDE 실험 한정: hypothetical_doc × 도서별 결과 샘플 추출 ──────────────────
#
# hypothetical_doc  → run.summary (세션당 1개)
# title/book_intro/score → run의 results_table artifact (책당 1행)

api  = wandb.Api()
runs = api.runs(WANDB_PROJECT)

sample_rows = []

for run in runs:
    if run.config.get("query_transform") != "hyde":
        continue

    hypo_doc     = run.summary.get("hypothetical_doc", "")
    persona_name = run.config.get("persona_name", run.summary.get("persona_name", ""))
    session_id   = run.summary.get("session_id", "")

    # wandb Table은 history가 아니라 logged_artifacts에서 꺼내야 함
    table_df = None
    for artifact in run.logged_artifacts():
        if artifact.type != "run_table":
            continue
        try:
            table = artifact.get("results_table")
            if table is not None:
                table_df = pd.DataFrame(table.data, columns=table.columns)
                break
        except Exception:
            continue

    if table_df is None:
        continue

    for _, row in table_df.iterrows():
        sample_rows.append({
            "run_id":           run.id,
            "session_id":       session_id,
            "persona_name":     persona_name,
            "hypothetical_doc": hypo_doc,
            "title":            row.get("title", ""),
            "author":           row.get("author", ""),
            "category":         row.get("category", ""),
            "book_intro":       row.get("book_intro", ""),
            "score":            int(row.get("score", 0)),
            "reason":           row.get("reason", ""),
        })

if not sample_rows:
    print("⚠️  수집된 데이터 없음.")
    df_hyde_sample = pd.DataFrame(columns=["run_id", "session_id", "persona_name", "hypothetical_doc",
                                           "title", "author", "category", "book_intro", "score", "reason"])
else:
    df_hyde_sample = pd.DataFrame(sample_rows)
    print(f"총 행 수: {len(df_hyde_sample)}  (런: {df_hyde_sample['run_id'].nunique()}, 페르소나: {df_hyde_sample['persona_name'].nunique()})")

df_hyde_sample.head(10)

wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downlo

총 행 수: 450  (런: 45, 페르소나: 3)


,run_id,session_id,persona_name,hypothetical_doc,title,author,category,book_intro,score,reason
0,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",몸값을 높여라 - 지금 당신의 가치에 적합한 연봉을 받고 있는가?,[히라이 다카시],성공,지금 당신의 커리어는 어떤 수준인가? 요즘은 경제 환경이 유난히 좋지 않다. 사회와...,1,커리어 설계와 비즈니스 스킬 획득을 통해 이직 준비에 도움이 될 수 있는 내용이 포...
1,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",지금 회사와 안녕하고 싶은 날,[예동희],CEO/비즈니스맨을 위한 능력계발,"지금, 이직을 해야 할까?, 내가 가고 싶은 곳에 갈 수 있을까?, 회사 지원은 어...",1,이직에 관한 다양한 질문과 경험을 다루어 이직 준비에 실질적인 도움이 될 수 있습니다.
2,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",마음에 꿈을 크게 그려라,[유희태],기독교(개신교),"그동안 기업현장 위주로 강의한 것을 토대로, 나름대로 어려웠던 부분과 고민하면서 해...",0,"기업 현장에서의 경험을 다루지만, 이직 준비와 관련된 구체적인 통찰이 부족합니다."
3,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",너만 힘드냐? 나도 힘들어 죽겠다.,[최웅섭],성공,이 책은 단지 개인적인 성찰과 성장에 머물지 않는다. 우리는 서로의 삶 속에서 영감...,0,개인적인 성찰과 성장에 중점을 두고 있어 이직 준비와는 직접적인 연관이 없습니다.
4,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",1%의 가능성을 성공으로 바꾼 사람들,[이대희],성공,"직장을 잃고, 상처를 받고, 갑작스러운 사고와 병으로 고통당하고, 꿈을 잃고 방황하...",0,"성공 사례를 다루지만, 이직 준비와 관련된 구체적인 조언이 부족합니다."
5,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",설래는 취업 준비 - 시작부터 바르게,[우설래],취업/상식/적성,"5,000여 건의 상담을 통해 취준생들이 자신의 삶 전반에 대해 깊이 고민하고 원하...",1,취업 준비를 위한 체계적인 프로세스를 제공하여 이직 준비에 실질적인 도움이 될 수 ...
6,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",재개발ㆍ재건축 감정평가론,[이철현],경상계열,저자가 실무에서 익히고 배운 노하우가 고스란히 녹아있다. 우선 그는 각 업무의 개념...,0,전문적인 내용으로 이직 준비와는 관련이 없습니다.
7,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",셀프 코칭의 기술 - 포춘 100대 기업 CEO와 임원진이 활용하는 자기 주도 성공법,[이정숙],성공,"적절한 시기의 현명한 조언은 사람을 성장시키는 원동력이 되는데, 가능하다면 나를 가...",1,자기 주도적인 성장 방법을 제시하여 이직 준비에 도움이 될 수 있습니다.
8,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",고용불안에도 흔들리지 않는 힘! 커리어 GPS - 두 번째 일자리를 위한 플랜 B를...,"[김경희, 김소현, 이민아]",중년의 자기계발,"내일이라도, 직장을 잃을 수 있다 회사를 다녀도 불안한 직장인, 이제는 나만의 힘을...",1,커리어 관리와 전직에 대한 핵심적인 내용을 제공하여 이직 준비에 유용합니다.
9,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용...",금융권 취업 가이드 - 금융 리더를 꿈꾸는 20대를 위한,[신재민],취업/진로/유망직업,취업 준비생 중에서도 특히 금융권 취업을 목표로 하는 사람을 위한 내용을 담고 있다...,0,"금융권에 특화된 내용으로, 일반적인 이직 준비와는 관련이 없습니다."


In [23]:
pd.set_option("display.max_colwidth", None)
df_hyde_sample[df_hyde_sample["persona_name"] == "A_최재원"].reset_index(drop=True).head(3)

,run_id,session_id,persona_name,hypothetical_doc,title,author,category,book_intro,score,reason
0,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용적인 팁을 담고 있습니다. 짧고 간결한 챕터로 구성되어 있어, 바쁜 일상 속에서도 손쉽게 읽을 수 있습니다. 저자는 개인의 실패와 성공을 통해 얻은 통찰을 공유하며, 독자가 자신의 커리어 방향을 재정립할 수 있도록 돕습니다. 긍정적인 결말로 독자의 불안감을 해소하고, 새로운 시작을 위한 희망의 메시지를 전합니다. 실생활에 바로 적용 가능한 조언으로, 커리어 전환에 대한 두려움을 덜어줄 이 책을 통해 자신만의 길을 찾아보세요.",몸값을 높여라 - 지금 당신의 가치에 적합한 연봉을 받고 있는가?,[히라이 다카시],성공,지금 당신의 커리어는 어떤 수준인가? 요즘은 경제 환경이 유난히 좋지 않다. 사회와 자신의 앞날이 보이지 않아 흘러가는 대로 따라가고는 있지만 어떻게든 방법을 찾아야 한다. 는 위기감을 느끼는 사람이 많다. 하지만 그저 시대의 흐름에 내맡기기만 하면 아무것도 바뀌지 않는다. 현재의 고민을 해소하고 자신의 인생을 개척해 나가기 위해서는 먼저 작은 걸음이라도 한 발짝을 내딛는 것이 중요하다. 그 첫 걸음이 바로 자신의 커리어를 진지하게 고민하는 일이다. 커리어를 쌓기 위해서는 비즈니스 스킬과 경영에 관한 최소한의 지식을 갖출 필요가 있다. 이 책에는 독자들이 커리어 설계 와 비즈니스 스킬 획득 이라는 두 마리 토끼를 동시에 잡을 수 있도록 구성하였다.,1,커리어 설계와 비즈니스 스킬 획득을 통해 이직 준비에 도움이 될 수 있는 내용이 포함되어 있습니다.
1,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용적인 팁을 담고 있습니다. 짧고 간결한 챕터로 구성되어 있어, 바쁜 일상 속에서도 손쉽게 읽을 수 있습니다. 저자는 개인의 실패와 성공을 통해 얻은 통찰을 공유하며, 독자가 자신의 커리어 방향을 재정립할 수 있도록 돕습니다. 긍정적인 결말로 독자의 불안감을 해소하고, 새로운 시작을 위한 희망의 메시지를 전합니다. 실생활에 바로 적용 가능한 조언으로, 커리어 전환에 대한 두려움을 덜어줄 이 책을 통해 자신만의 길을 찾아보세요.",지금 회사와 안녕하고 싶은 날,[예동희],CEO/비즈니스맨을 위한 능력계발,"지금, 이직을 해야 할까?, 내가 가고 싶은 곳에 갈 수 있을까?, 회사 지원은 어떻게 해야 하지?, 오랜만에 쓰는 이력서, 경력직은 어떻게 써야하나?, 경력자의 면접은 무엇이 다를까?, 합격하고 나면 사표도 써야 하고 연봉협상도 해야 할 텐데. 등 직장인이라면 한 번쯤 고민해보았을짐한 질문들에 이직 선배들이 답한다. 이직 선배들의 멘토링 스토리를 보며 이직을 생각하는 순간부터, 이직을 준비하고, 경험하고, 마무리한 그 이후까지 당신이 겪게 될 이직에 관한 모든 것 을 미리 체험해 볼 수 있는 기회를 얻게 될 것이다.",1,이직에 관한 다양한 질문과 경험을 다루어 이직 준비에 실질적인 도움이 될 수 있습니다.
2,0ss17vyd,8fa61928,A_최재원,"이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용적인 팁을 담고 있습니다. 짧고 간결한 챕터로 구성되어 있어, 바쁜 일상 속에서도 손쉽게 읽을 수 있습니다. 저자는 개인의 실패와 성공을 통해 얻은 통찰을 공유하며, 독자가 자신의 커리어 방향을 재정립할 수 있도록 돕습니다. 긍정적인 결말로 독자의 불안감을 해소하고, 새로운 시작을 위한 희망의 메시지를 전합니다. 실생활에 바로 적용 가능한 조언으로, 커리어 전환에 대한 두려움을 덜어줄 이 책을 통해 자신만의 길을 찾아보세요.",마음에 꿈을 크게 그려라,[유희태],기독교(개신교),"그동안 기업현장 위주로 강의한 것을 토대로, 나름대로 어려웠던 부분과 고민하면서 해결해야만 했던 사항들을 관심 있는 분들과 후배들에게 조금이나마 도움이 되었으면 하는 마음을 담은 책. 삶 속에서 도전과 창의, 열정적으로 활동했던 내용들을 이전 책 마음에 꿈을 그려라 에 진솔하게 소개했다.",0,"기업 현장에서의 경험을 다루지만, 이직 준비와 관련된 구체적인 통찰이 부족합니다."


In [ ]:
# # ── 진단: hyde 런이 실제로 몇 개인지, config 구조가 어떤지 확인 ──────────────
# api  = wandb.Api()
# runs = api.runs(WANDB_PROJECT)

# total = 0
# hyde_count = 0
# for run in runs:
#     total += 1
#     qt = run.config.get("query_transform", "__missing__")
#     if total <= 3:
#         print(f"[샘플 런] id={run.id}, config keys={list(run.config.keys())}, query_transform={qt!r}")
#     if qt == "hyde":
#         hyde_count += 1
#         hypo = run.summary.get("hypothetical_doc", "__missing__")
#         print(f"  → hyde 런 발견: id={run.id}, hypothetical_doc={repr(hypo)[:80]}")

# print(f"\n총 런: {total}, hyde 런: {hyde_count}")

[샘플 런] id=a87yu0i3, config keys=['run_index', 'persona_name', 'collection_name', 'query_transform', 'use_genre_filter'], query_transform='none'
[샘플 런] id=rg72fnm7, config keys=['run_index', 'persona_name', 'collection_name', 'query_transform', 'use_genre_filter'], query_transform='none'
[샘플 런] id=lxug357a, config keys=['run_index', 'persona_name', 'collection_name', 'query_transform', 'use_genre_filter'], query_transform='none'
  → hyde 런 발견: id=0ss17vyd, hypothetical_doc='이직 준비의 길잡이가 되어줄 이 책은, 실제 경험을 바탕으로 한 생생한 사례와 실용적인 팁을 담고 있습니다. 짧고 간결한 챕터로 구성되어 있어
  → hyde 런 발견: id=7f1ns6uj, hypothetical_doc='이 책은 현대 사회에서 변화와 혁신을 이끄는 원동력을 탐구하며, 직장 내 성공의 비결을 실질적인 사례를 통해 제시합니다. 경제적 원리와 철학적
  → hyde 런 발견: id=eo5c69yv, hypothetical_doc='직장 생활의 새로운 전환점을 찾고 있는 독자를 위해 집필된 이 책은 경제적 통찰과 심리적 접근을 결합하여 커리어 방향을 재조명하는 데 필요한 
  → hyde 런 발견: id=kho5z432, hypothetical_doc='이 책은 개인의 커리어 재정립을 위한 필독서로, 심리적 깊이와 실용성을 완벽히 결합했습니다. 독자는 저자의 진솔한 경험담을 통해 이직 준비 과
  → hyde 런 발견: id=79eljkky, hypothetical_doc='이 책은 경제와 심리의 교차점에서